### Search Engine With Tools And Agents

In [ ]:
## Arxiv -- research
## tool creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [ ]:
# %pip install wikipedia
# %pip install arxiv


In [ ]:

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

In [ ]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,docs_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv)
arxiv.name

In [ ]:
tools=[wiki,arxiv]

In [ ]:
## Custome tools[RAG Tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [ ]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,embedding)
retriver=vectordb.as_retriever()
retriver

In [ ]:
from langchain.tools.retriever import create_retriever_tool
retriver_tool = create_retriever_tool(
	retriver, 
	"langsmith-search", 
	"search any information about Langsmith"
)

retriver_tool.name

In [ ]:
tools=[wiki,arxiv,retriver_tool]

In [ ]:
tools

In [ ]:
## run all these tools with llm and agents

## tools : llm  --> agentExecutor

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile")



In [ ]:
## prompt template
from langchain import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages
# from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant."),
#     MessagesPlaceholder("chat_history", optional=True),
#     ("human", "{input}"),
#     MessagesPlaceholder("agent_scratchpad"),
# ])

In [ ]:
## agents
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

In [ ]:
## agent executor
from langchain.agents import AgentExecutor
agent_executor= AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

In [ ]:
agent_executor.invoke({"input":"tell me about langsmith"})

In [ ]:
from langchain_core.tools import Tool

def wiki_tool(query: str) -> str:
    return f"Wiki answer for: {query}"

def arxiv_tool(query: str) -> str:
    return f"Arxiv papers summary for: {query}"

def retriever_tool(query: str) -> str:
    return f"Custom processing result: {query}"
# Tool function return a string 
def w(query: str) -> str:
    # process query
    return f"Answer for: {query}"

# Define tools using LangChain Tool class
ml_tool = Tool(
    name="wiki",
    func=w,
    description="Returns answer about ML"
)
tools = [
    Tool(name="wiki", func=wiki_tool, description="Search Wikipedia for information"),
    Tool(name="arxiv", func=arxiv_tool, description="Search Arxiv papers for relevant results"),
    Tool(name="custom", func=retriever_tool, description="Use custom logic or database")
]

from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools=tools,
    llm=llm,  # your Groq or other LLM
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("tell me about artificial intelligence")
print(response)
